# Scenario 05 — Vault Lifecycle

Full vault cycle: deposit → lock → unbond → wait → claim. Vault operations require an
explicit `signer=` (no env-var fallback per decision C1-vault) to prevent accidental
deposits/withdrawals from the wrong account.

**Role:** Any (client, provider, or master can all hold vault balances).

**Prerequisites:**
- A funded testnet wallet — private key in `VAULT_ACTOR_KEY`
- Enough native token for deposits + gas
- **Unbonding period wait**: step 5 requires waiting `Vault.UNBONDING_PERIOD` seconds
  before `claim()` succeeds. Check the current value first.

**Flow:**

```
Vault actor
  │
  ├─ 1. deposit              → balance_of ↑
  ├─ 2. lock                 → lockup_of ↑, balance_of ↓
  ├─ 3. (snapshot via Provider for readability)
  ├─ 4. unbond               → unbonding_of populated
  ├─ 5. wait UNBONDING_PERIOD
  └─ 6. claim                → balance_of ↑
```


## 0. Setup


In [ ]:
import os
import time

from web3 import Web3

from ogpu import ChainConfig, ChainId
from ogpu.protocol import Provider, vault

ChainConfig.set_chain(ChainId.OGPU_TESTNET)

VAULT_ACTOR_KEY = os.environ["VAULT_ACTOR_KEY"]
from eth_account import Account
VAULT_ACTOR_ADDR = Account.from_key(VAULT_ACTOR_KEY).address
print(f"Vault actor: {VAULT_ACTOR_ADDR}")


## 1. Deposit


In [ ]:
DEPOSIT = Web3.to_wei(0.5, "ether")

r = vault.deposit(VAULT_ACTOR_ADDR, DEPOSIT, signer=VAULT_ACTOR_KEY)
print(f"Deposit tx: {r.tx_hash}")
print(f"Balance: {vault.get_balance_of(VAULT_ACTOR_ADDR)} wei")


## 2. Lock (stake a portion)


In [ ]:
LOCK = Web3.to_wei(0.2, "ether")

r = vault.lock(LOCK, signer=VAULT_ACTOR_KEY)
print(f"Lock tx: {r.tx_hash}")

print(f"Balance: {vault.get_balance_of(VAULT_ACTOR_ADDR)}")
print(f"Lockup : {vault.get_lockup_of(VAULT_ACTOR_ADDR)}")


## 3. Snapshot via `Provider`

If the actor is a registered provider, wrapping the address in `Provider(addr).snapshot()`
grabs every vault + terminal field in one logical batch — great for dashboards.


In [ ]:
# Works for any address, not just registered providers — reads still succeed
p = Provider(VAULT_ACTOR_ADDR)
print(f"Balance        : {p.get_balance()}")
print(f"Lockup         : {p.get_lockup()}")
print(f"Total earnings : {p.get_total_earnings()}")
print(f"Is eligible    : {p.is_eligible()}")


## 4. Unbond


In [ ]:
UNBOND = Web3.to_wei(0.1, "ether")

r = vault.unbond(UNBOND, signer=VAULT_ACTOR_KEY)
print(f"Unbond tx: {r.tx_hash}")

print(f"Unbonding       : {vault.get_unbonding_of(VAULT_ACTOR_ADDR)}")
print(f"Unbonding ts    : {vault.get_unbonding_timestamp(VAULT_ACTOR_ADDR)}")
print(f"Unbonding period: {vault.get_unbonding_period()} seconds")


## 5. Wait

The unbonding period is enforced on-chain. Until the full wait elapses, `claim()` reverts.
Fetch the period and wait — or come back later and run the next cell.


In [ ]:
period = vault.get_unbonding_period()
print(f"Waiting {period} seconds for unbonding to mature...")
time.sleep(period + 5)
print("Done waiting.")


## 6. Claim


In [ ]:
r = vault.claim(signer=VAULT_ACTOR_KEY)
print(f"Claim tx: {r.tx_hash}")
print(f"Final balance: {vault.get_balance_of(VAULT_ACTOR_ADDR)}")
